# Phase 01 — Scraping Berita 4 Bank (BCA, BRI, BNI, Mandiri) dari CNBC Indonesia

Notebook **mandiri (self-contained)**. Mengumpulkan berita ekonomi 4 bank besar dari CNBC Indonesia.

### Sumber data = API pencarian CNBC (JSON)
Halaman pencarian biasa (`/search?...`) kini **dirender JavaScript**, sehingga `requests` + `BeautifulSoup` tidak melihat daftar artikel. Karena itu kita pakai **API pencarian CNBC** yang mengembalikan **JSON**:

```
https://www.cnbcindonesia.com/api/v2/search?query=<kw>&fromdate=YYYY/MM/DD&todate=YYYY/MM/DD&start=<N>&limit=<M>
```

Satu response sudah berisi **judul, tanggal, URL, dan isi artikel (`strisi`)** sekaligus. Paginasi lewat `start` (offset) + `limit`.

### Cara pakai
- **Nilai penentu ada LANGSUNG di dalam pemanggil `jalankan(...)`**, ditandai `# <- bisa diubah`.
- Filter 4 bank lewat **keyword** (nama + kode saham); hasil **digabung & dedupe** via `id_artikel`.
- **Periode:** 20 Oktober 2024 – 20 Oktober 2026 (batas waktu untuk membatasi volume data; tanpa klaim kausalitas).
- **Checkpoint/resume:** hasil ditulis *append* ke CSV; `id_artikel` yang sudah ada dilewati.
- Isi disimpan mentah (cleaning & pemecahan kalimat di Phase 02).
- **Kanal:** default semua kanal (`kanal_dipakai=None`); kolom kanal tidak ditulis ke CSV.
- **Kolom CSV:** `id_artikel, keyword, url, timestamp (YYYY-MM-DD), judul, isi_mentah`.

> Catatan etika: data untuk riset akademik non-komersial; patuhi Ketentuan Penggunaan CNBC Indonesia.

## 1. Import

In [8]:
import re
import csv
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

## 2. Utilitas jaringan & pembersih isi

`fetch_json()` mengunduh JSON dari API dengan retry. `bersikan_html()` mengubah isi artikel (dari field `strisi`, berupa HTML) menjadi teks bersih.

In [9]:
def fetch_json(url): # logic pengambilan data pemnggunakan API
    headers = {'User-Agent': 'Chrome/120.0.0.0'}
    for percobaan in range(1, 4):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (404, 410):
                return None
        except (requests.RequestException, ValueError) as e:
            print(f"    [ERR] {percobaan}/3: {type(e).__name__}")
        time.sleep(2 ** percobaan)           # backoff: 2s, 4s, 8s
    return None

def bersikan_html(html): # Convert hasil scrawl/scrape API menjadi html
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")
    for buang in soup.find_all(["script", "style", "iframe"]):
        buang.decompose()
    for v in soup.find_all("a", class_=re.compile(r"autovideocnbc")):
        v.decompose()
    for b in soup.find_all(class_=re.compile(r"lihatjg|linksisip")):   # buang blok 'Baca juga' (judul artikel lain yang nyasar)
        b.decompose()
    for s in soup.find_all("strong"):   # buang dateline 'Kota, CNBC Indonesia' (dash floating dibersihkan di bawah)
        if re.search(r"CNBC\s*(?:&nbsp;|\s)*Indonesia", s.get_text()):
            s.decompose()
            break
    for p in soup.find_all(["fieldset", "p", "div"]):   # buang boilerplate Disclaimer/Sanggahan (sumber duplikat)
        if re.match(r"\s*-?\s*(Disclaimer|Sanggahan)\b", p.get_text(" ", strip=True), flags=re.I):
            p.decompose()
    teks = soup.get_text(" ", strip=True)
    teks = re.sub(r"^\s*[-\u2013]\s*", "", teks)   # buang dash floating sisa dateline di awal
    teks = re.sub(r"-?\s*(Disclaimer|Sanggahan)\s*:.*$", "", teks, flags=re.I)   # jaga-jaga: buang sisa disclaimer di ekor teks
    teks = re.sub(r"\[Gambas:[^\]]*\]", " ", teks) # filter gambar
    teks = re.sub(r"\(\s*[a-z]{2,4}\s*/\s*[a-z]{2,4}\s*\)\s*$", "", teks).strip() # filter penulis
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

## 3. Ambil artikel dari API pencarian

`api_url()` membangun URL API. `ambil_dari_api()` menelusuri hasil satu keyword via `start`/`limit`, mengambil tiap artikel **teks** (membuang video), beserta judul, tanggal (YYYY-MM-DD), URL, dan isi.

In [10]:
def api_url(keyword, fromdate, todate, start, limit):
    kw = keyword.replace(" ", "+")
    return (f"https://www.cnbcindonesia.com/api/v2/search?query={kw}&fromdate={fromdate}&todate={todate}&start={start}&limit={limit}")

# Pola URL artikel CNBC: /{kanal}/{YYYYMMDDhhmmss}-{kode_tipe}-{id}/{slug}
pola_url = re.compile(r"/(\w+)/(\d{14})-(\d+)-(\d+)/")

def ambil_dari_api(keyword, fromdate, todate, maks_artikel, jeda, kanal_dipakai): # Hasil scrape API, diubah menjadi disc
    limit = 100 # Limit 100 artikel per API
    hasil = []
    start = 0
    while start < maks_artikel:
        data = fetch_json(api_url(keyword, fromdate, todate, start, limit))
        time.sleep(jeda) # lag per scrape
        if not data or not data.get("data"):
            break

        for it in data["data"]:
            url = it.get("articleUrl") or it.get("url") or ""
            m = pola_url.search(url)
            if not m:
                continue
            kanal, ts, kode_tipe, id_artikel = m.groups()
            if kanal_dipakai and kanal not in kanal_dipakai:   # filter kanal (None = semua)
                continue

            if kode_tipe in ("18", "19"): # kode video
                continue

            judul = (it.get("strjudul") or "").strip()
            if judul.lower().startswith("video"): # tidak ambil artikel video
                continue

            if url.startswith("/"):
                url = "https://www.cnbcindonesia.com" + url
            tanggal = f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]}"

            hasil.append({
                "id_artikel": id_artikel, "url": url, "timestamp": tanggal,
                "judul": judul, "isi": bersikan_html(it.get("strisi", "")),
            })
        if len(data["data"]) < limit: # halaman terakhir
            break
        start += limit
    return hasil

## 4. Jalankan scraping (dengan checkpoint & resume)

Tiap artikel ditulis *append* ke CSV; `id_artikel` yang sudah ada dilewati (resume + dedupe lintas keyword).

In [11]:
def jalankan(keywords, fromdate, todate, maks_artikel, jeda, output_csv, kanal_dipakai):
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    header = ["id_artikel", "keyword", "url", "timestamp", "judul", "isi_mentah"]

    # resume + dedupe: kumpulkan id_artikel yang sudah tersimpan
    sudah = set()
    if output_csv.exists():
        with open(output_csv, "r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                sudah.add(str(row.get("id_artikel", "")))
    print(f"[RESUME] {len(sudah)} artikel sudah ada, akan dilewati.\n") # Check jika data sudah pernah discrape)

    file_baru = not output_csv.exists()
    f = open(output_csv, "a", encoding="utf-8", newline="", buffering=1)
    writer = csv.DictWriter(f, fieldnames=header)
    if file_baru:
        writer.writeheader()

    n_baru = 0
    for keyword in keywords:
        artikel = ambil_dari_api(keyword, fromdate, todate, maks_artikel, jeda, kanal_dipakai)
        print(f"=== keyword '{keyword}': {len(artikel)} artikel dari API ===")
        for a in artikel:
            if a["id_artikel"] in sudah:
                continue   # sudah pernah -> dedupe
                
            writer.writerow({
                "id_artikel": a["id_artikel"], "keyword": keyword, "url": a["url"],
                "timestamp": a["timestamp"], "judul": a["judul"], "isi_mentah": a["isi"],
            })
            
            sudah.add(a["id_artikel"])
            n_baru += 1
            print(f"  [{keyword}] +{a['judul'][:60]}")

    f.close()
    print()
    print(f"selesai, {n_baru} artikel baru ditambahkan sesi ini -> {output_csv}")
    return output_csv

In [12]:
output_csv = jalankan(
    keywords=[
        "BCA", "BBCA", "BRI", "BBRI",
        "BNI", "BBNI", "Mandiri", "BMRI",
    ],
    fromdate="2024/10/20", 
    todate="2026/10/20",
    maks_artikel=2000,
    jeda=1.0,
    output_csv="data/raw/raw.csv",
    kanal_dipakai="market",
)

[RESUME] 0 artikel sudah ada, akan dilewati.

=== keyword 'BCA': 475 artikel dari API ===
  [BCA] +Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA Paling Banyak 
  [BCA] +RI Alami Halusinasi Daya Beli, Ekonom Ungkap Faktanya!
  [BCA] +BCA Mau Buyback 19-21 Oktober, Karyawan Dilarang Jual-Beli S
  [BCA] +Asing Keluar Rp2,92 Triliun di Sesi I, Saham BCA Paling Bany
  [BCA] +Ada Bank Yang Sudah Jual Dolar di Rp18.300
  [BCA] +Rumor Haji Isam Caplok BCA, Ini Pernyataan OJK
  [BCA] +Investor Kini Maunya Sebentar Pegang Surat Utang RI, Ini Pen
  [BCA] +Investor Pilih Berebut Surat Utang Tenor Pendek, Ragukan Eko
  [BCA] +Investor Asing Diam-diam Net Sell Rp166 Miliar Sepanjang Ses
  [BCA] +BI Mau Kartu Kredit RI Lebih Berdaya, Ini Target Destry
  [BCA] +Asing Diam-Diam Lepas 10 Saham Ini Kala IHSG Terkoreksi
  [BCA] +Kemenkeu Gelar Lelang Sukuk Hari ini, Target Raup Rp10 Trili
  [BCA] +Rupiah Ditutup Melemah 0,15%, Nilai Tukar Dolar AS Tembus Rp
  [BCA] +Waspada! Rupiah Terus Keok Lawan Dolar 

## 5. Verifikasi hasil

In [8]:
df = pd.read_csv(output_csv, dtype={"id_artikel": str})
print(f"Total artikel : {len(df)}")
print(f"id unik       : {df['id_artikel'].nunique()} (duplikat: {len(df)-df['id_artikel'].nunique()})")
print(f"rata panjang isi: {df['isi_mentah'].str.len().mean():.0f} char")
print("\nJumlah per keyword:")
print(df["keyword"].value_counts())
df[["id_artikel", "keyword", "timestamp", "judul"]].head(10)

Total artikel : 7190
id unik       : 7190 (duplikat: 0)
rata panjang isi: 3471 char

Jumlah per keyword:
keyword
BBCA       1734
BRI        1693
Mandiri    1298
BCA         788
BNI         720
BBRI        435
BMRI        281
BBNI        241
Name: count, dtype: int64


,id_artikel,keyword,timestamp,judul
0,777827,BCA,2026-10-20,"Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ..."
1,777666,BCA,2026-10-19,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F..."
2,777293,BCA,2026-10-19,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara..."
3,776992,BCA,2026-10-16,"Dulu Terbang Tinggi, Kini Ambruk! Bubble Saham..."
4,776824,BCA,2026-10-16,"Asing Keluar Rp2,92 Triliun di Sesi I, Saham B..."
5,776759,BCA,2026-10-16,Ada Bank Yang Sudah Jual Dolar di Rp18.300
6,776668,BCA,2026-10-16,"Rumor Haji Isam Caplok BCA, Ini Pernyataan OJK"
7,776295,BCA,2026-10-15,Investor Kini Maunya Sebentar Pegang Surat Uta...
8,776080,BCA,2026-10-14,Badan Intelijen-OJK Bagikan Tips agar HP Tak D...
9,776155,BCA,2026-10-14,Investor Pilih Berebut Surat Utang Tenor Pende...


In [13]:
# Contoh isi satu artikel (cek kualitas ekstraksi)
c = df[df["isi_mentah"].str.len() >= 150].iloc[0]
print("JUDUL :", c["judul"])
print("URL   :", c["url"])
print("-" * 70)
print(c["isi_mentah"][:1200], "...")

JUDUL : Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA Paling Banyak Dibuang
URL   : https://www.cnbcindonesia.com/market/20261020130533-17-777827/aksi-jual-asing-rp197-t-di-sesi-i-saham-bca-paling-banyak-dibuang
----------------------------------------------------------------------
Jakarta, CNBC Indonesia - Indeks Harga Saham Gabungan (IHSG) berhasil mengakhiri perdagangan sesi I hari ini, Selasa (20/10/2026), di zona hijau. IHSG ditutup di level 5.204,63, naik 208,02 poin atau 4,16%. Namun, asing masih saja mencatatkan penjualan dengan nilai yang cukup besar. Investor asing mencatatkan net sell sebesar Rp1,97 triliun di seluruh pasar pada akhir sesi I perdagangan Selasa ini. Mengutip Stockbit Sekuritas menunjukkan total pembelian asing mencapai Rp5,96 triliun, sedangkan penjualannya Rp7,93 triliun. Nilai transaksi dihitung menggunakan harga rata-rata sesi I. Baca: Banyak Saham Diborong, Akhir Sesi I IHSG Menguat 4,16% Tekanan jual asing terutama datang dari sektor perbankan. PT. Bank 

## 6. Menjalankan full-run (4 bank, 20 Okt 2024 – 20 Okt 2026)

Di sel pemanggil (bagian 4), ganti argumen:
- `keywords` → buka komentar daftar 8 keyword 4 bank.
- `fromdate="2024/10/20"`, `todate="2026/10/20"`.
- `output_csv="data/raw/cnbc_4bank_full.csv"`.

Lalu jalankan ulang. Hasil tiap keyword digabung & dedupe via `id_artikel`. Aman ditinggal: bila terputus, jalankan ulang — artikel yang sudah ada di CSV dilewati.